In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set DEBUG = True for rapid code verification (seconds), False for full training.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "query_samples": 32,
    "candidate_samples": 64,
    "epochs": 2,
    "batch_size": 4,
    "log_interval": 2,
    "lr": 1e-2,  # Higher learning rate standard for prefix tuning parameters
    "num_prefix_tokens": 16,
    "prefix_mid_dim": 512,
    "max_seq_len": 768,
    "temperature": 0.05
}

FULL_CONFIG = {
    "query_samples": None,  # Full dataset
    "candidate_samples": None,
    "epochs": 3,
    "batch_size": 8,
    "log_interval": 20,
    "lr": 1e-2,
    "num_prefix_tokens": 16,
    "prefix_mid_dim": 512,
    "max_seq_len": 1024,
    "temperature": 0.05
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Phase 7 Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

=== Phase 7 Execution Mode: PRODUCTION RUN ===
Configuration: {'query_samples': None, 'candidate_samples': None, 'epochs': 3, 'batch_size': 8, 'log_interval': 20, 'lr': 0.01, 'num_prefix_tokens': 16, 'prefix_mid_dim': 512, 'max_seq_len': 1024, 'temperature': 0.05}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_HYBRID = "A-240242/legal-hybridencoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_pcr_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_7_PCR/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 7 Prior Case Retrieval Data...")
raw_train_queries = load_pcr_file("Train/train.json")
train_candidates = load_pcr_file("Train/train_candidates.json")

raw_dev_queries = load_pcr_file("Dev/dev.json")
dev_candidates = load_pcr_file("Dev/dev_candidates.json")

raw_test_queries = load_pcr_file("Test/test.json")
test_candidates = load_pcr_file("Test/test_candidates.json")

def parse_queries(raw_data):
    """
    Normalizes query data into a standardized list of dicts regardless of 
    whether the input JSON is a dictionary (id -> query_obj) or a list.
    """
    if isinstance(raw_data, dict):
        if "queries" in raw_data and isinstance(raw_data["queries"], list):
            return raw_data["queries"]
        parsed = []
        for q_id, q_val in raw_data.items():
            if isinstance(q_val, dict):
                item = dict(q_val)
                item["id"] = str(q_id)
                parsed.append(item)
            else:
                parsed.append({"id": str(q_id), "text": q_val})
        return parsed
    elif isinstance(raw_data, list):
        return raw_data
    return []

def format_case_text(text_field):
    if isinstance(text_field, list):
        return " ".join(text_field)
    return str(text_field)

# Normalize queries into standard lists
train_queries = parse_queries(raw_train_queries)
dev_queries = parse_queries(raw_dev_queries)
test_queries = parse_queries(raw_test_queries)

print(f"Raw Loaded Counts -> Train Queries: {len(train_queries)}, Dev Queries: {len(dev_queries)}, Test Queries: {len(test_queries)}")

# Subsample if running in DEBUG mode
if DEBUG and CFG["query_samples"]:
    train_queries = train_queries[:CFG["query_samples"]]
    dev_queries = dev_queries[:CFG["query_samples"]]
    test_queries = test_queries[:CFG["query_samples"]]
    
    if isinstance(train_candidates, dict):
        train_cand_keys = list(train_candidates.keys())[:CFG["candidate_samples"]]
        train_candidates = {k: train_candidates[k] for k in train_cand_keys}
        
    if isinstance(dev_candidates, dict):
        dev_cand_keys = list(dev_candidates.keys())[:CFG["candidate_samples"]]
        dev_candidates = {k: dev_candidates[k] for k in dev_cand_keys}
        
    if isinstance(test_candidates, dict):
        test_cand_keys = list(test_candidates.keys())[:CFG["candidate_samples"]]
        test_candidates = {k: test_candidates[k] for k in test_cand_keys}
    
    print(f"\n[DEBUG] Subsampled -> Train Queries: {len(train_queries)}, Dev Queries: {len(dev_queries)}, Test Queries: {len(test_queries)}")

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

Phase_7_PCR/Dev/dev_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

Raw Loaded Counts -> Train Queries: 827, Dev Queries: 118, Test Queries: 237


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class PCRBiEncoderDataset(Dataset):
    """
    Dataset for Training Bi-Encoder Query-Candidate Pairs.
    Pairs each Query document with one of its positive relevant candidate documents.
    """
    def __init__(self, queries, candidates_dict, tokenizer, max_length=1024):
        self.samples = []
        
        for q in queries:
            q_id = q.get("id")
            q_text = format_case_text(q.get("text", ""))
            rel_cands = q.get("relevant_candidates", [])
            
            for c_id in rel_cands:
                if c_id in candidates_dict:
                    c_text = format_case_text(candidates_dict[c_id])
                    self.samples.append({
                        "query_id": q_id,
                        "query_text": q_text,
                        "candidate_id": c_id,
                        "candidate_text": c_text
                    })
                    
        # Fallback if no candidate matches found in debug slice
        if len(self.samples) == 0 and len(queries) > 0 and len(candidates_dict) > 0:
            first_c_id = list(candidates_dict.keys())[0]
            first_c_text = format_case_text(candidates_dict[first_c_id])
            for q in queries:
                self.samples.append({
                    "query_id": q.get("id"),
                    "query_text": format_case_text(q.get("text", "")),
                    "candidate_id": first_c_id,
                    "candidate_text": first_c_text
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def pcr_collate_fn(batch):
    query_texts = [item["query_text"] for item in batch]
    candidate_texts = [item["candidate_text"] for item in batch]
    
    q_tok = tokenizer(
        query_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt"
    )
    c_tok = tokenizer(
        candidate_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt"
    )
    
    return {
        "query_input_ids": q_tok["input_ids"],
        "query_attention_mask": q_tok["attention_mask"],
        "candidate_input_ids": c_tok["input_ids"],
        "candidate_attention_mask": c_tok["attention_mask"]
    }

print("Building Phase 7 Pair DataLoader...")
train_dataset = PCRBiEncoderDataset(train_queries, train_candidates, tokenizer, max_length=CFG["max_seq_len"])
train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=pcr_collate_fn)

print(f"Created {len(train_dataset)} training query-positive candidate pair samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building Phase 7 Pair DataLoader...
Created 5308 training query-positive candidate pair samples.


In [4]:
class LayerNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(variance + self.eps) * self.weight + self.bias


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    cos = cos[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    sin = sin[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class HybridBidirectionalAttentionWithPrefix(nn.Module):
    """Bidirectional Multi-Head Self-Attention with Layer-wise Prefix Insertion."""
    def __init__(self, hidden_size=768, num_heads=12):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(
        self, 
        x: torch.Tensor, 
        attention_mask: torch.Tensor = None, 
        cos: torch.Tensor = None, 
        sin: torch.Tensor = None,
        prefix_k: torch.Tensor = None, # [B, num_heads, num_prefix, head_dim]
        prefix_v: torch.Tensor = None  # [B, num_heads, num_prefix, head_dim]
    ):
        B, S, _ = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            
        # Concatenate virtual Key/Value prefixes if present
        if prefix_k is not None and prefix_v is not None:
            k = torch.cat([prefix_k, k], dim=2) # [B, num_heads, P + S, head_dim]
            v = torch.cat([prefix_v, v], dim=2) # [B, num_heads, P + S, head_dim]
            
            # Extend attention mask for prefix tokens
            if attention_mask is not None:
                num_prefix = prefix_k.size(2)
                prefix_mask = torch.ones(B, num_prefix, device=x.device, dtype=attention_mask.dtype)
                attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)
                
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            attn_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + attn_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        
        return self.o_proj(attn_out)


class HybridConvFFN(nn.Module):
    """1D Depthwise-Convolutional Feed-Forward Network."""
    def __init__(self, hidden_size=768, intermediate_size=2048, kernel_size=3):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=intermediate_size,
            out_channels=intermediate_size,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=intermediate_size,
            bias=False
        )
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gated = F.silu(self.gate_proj(x)) * self.up_proj(x)
        gated_conv = gated.transpose(1, 2)
        gated_conv = self.conv1d(gated_conv).transpose(1, 2)
        return self.down_proj(gated_conv)


class HybridEncoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = LayerNorm(hidden_size)
        self.self_attn = HybridBidirectionalAttentionWithPrefix(hidden_size, num_heads)
        self.post_attention_layernorm = LayerNorm(hidden_size)
        self.mlp = HybridConvFFN(hidden_size, intermediate_size)

    def forward(
        self, 
        x: torch.Tensor, 
        attention_mask: torch.Tensor = None, 
        cos: torch.Tensor = None, 
        sin: torch.Tensor = None,
        prefix_k: torch.Tensor = None,
        prefix_v: torch.Tensor = None
    ):
        residual = x
        x = self.input_layernorm(x)
        x = self.self_attn(x, attention_mask=attention_mask, cos=cos, sin=sin, prefix_k=prefix_k, prefix_v=prefix_v)
        x = residual + x
        
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = residual + x
        return x


class HybridEncoderBackbone(nn.Module):
    """Hybrid Encoder Backbone supporting Prefix Tuning."""
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            HybridEncoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm = LayerNorm(hidden_size)
        self.rotary_emb = RotaryEmbedding(hidden_size // num_heads)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, layer_prefixes: torch.Tensor = None):
        """
        layer_prefixes: Optional tensor of shape [B, num_layers, 2, num_heads, num_prefix, head_dim]
        """
        B, S = input_ids.shape
        x = self.embed_tokens(input_ids)
        cos, sin = self.rotary_emb(x, S)
        
        for i, layer in enumerate(self.layers):
            pk = layer_prefixes[:, i, 0] if layer_prefixes is not None else None
            pv = layer_prefixes[:, i, 1] if layer_prefixes is not None else None
            x = layer(x, attention_mask=attention_mask, cos=cos, sin=sin, prefix_k=pk, prefix_v=pv)
            
        x = self.norm(x)
        return x

print("Custom Hybrid Encoder Backbone with Prefix Tuning written from scratch successfully.")

Custom Hybrid Encoder Backbone with Prefix Tuning written from scratch successfully.


In [5]:
class PrefixTunedHybridEmbedder(nn.Module):
    """
    Prefix Tuning Wrapper for Hybrid Encoder Dense Retrieval.
    Optimizes a soft reparameterized MLP network generating layer-wise K/V prefixes.
    """
    def __init__(self, backbone: HybridEncoderBackbone, num_prefix_tokens: int = 16, prefix_mid_dim: int = 512):
        super().__init__()
        self.backbone = backbone
        self.num_prefix_tokens = num_prefix_tokens
        
        num_layers = backbone.num_layers
        num_heads = backbone.num_heads
        head_dim = backbone.head_dim
        hidden_size = num_heads * head_dim
        
        # Trainable Prefix Parameters & Reparameterization Network
        self.prefix_embedding = nn.Embedding(num_prefix_tokens, hidden_size)
        self.prefix_mlp = nn.Sequential(
            nn.Linear(hidden_size, prefix_mid_dim),
            nn.Tanh(),
            nn.Linear(prefix_mid_dim, num_layers * 2 * num_heads * head_dim)
        )

    def get_layer_prefixes(self, batch_size: int, device: torch.device):
        prefix_tokens = torch.arange(self.num_prefix_tokens, device=device).unsqueeze(0).expand(batch_size, -1)
        prefix_embeds = self.prefix_embedding(prefix_tokens) # [B, num_prefix, hidden_size]
        
        # Project through reparameterization MLP
        past_key_values = self.prefix_mlp(prefix_embeds) # [B, num_prefix, num_layers * 2 * num_heads * head_dim]
        
        # Reshape to [B, num_layers, 2, num_heads, num_prefix, head_dim]
        past_key_values = past_key_values.view(
            batch_size,
            self.num_prefix_tokens,
            self.backbone.num_layers,
            2,
            self.backbone.num_heads,
            self.backbone.head_dim
        ).permute(0, 2, 3, 4, 1, 5)
        
        return past_key_values

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        B = input_ids.size(0)
        layer_prefixes = self.get_layer_prefixes(B, input_ids.device)
        
        hidden_states = self.backbone(input_ids, attention_mask=attention_mask, layer_prefixes=layer_prefixes)
        
        # Mean Pooling over token representations
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_hidden = (hidden_states * mask_expanded).sum(dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            pooled = sum_hidden / sum_mask
        else:
            pooled = hidden_states.mean(dim=1)
            
        # L2 Normalize Embeddings
        return F.normalize(pooled, p=2, dim=-1)


# 1. Instantiate Base Backbone Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone = HybridEncoderBackbone(
    vocab_size=len(tokenizer), 
    hidden_size=768, 
    num_layers=6, 
    num_heads=12, 
    intermediate_size=2048
)

# 2. Fetch and Load Phase 1 Pre-trained Weights
print(f"Fetching Phase 1 Checkpoint Weights from '{HF_MODEL_REPO_HYBRID}'...")
try:
    weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_HYBRID, filename="model.safetensors", token=HF_TOKEN)
    state_dict = load_file(weight_path)
    missing, unexpected = backbone.load_state_dict(state_dict, strict=False)
    print(f"Loaded Phase 1 weights successfully from '{HF_MODEL_REPO_HYBRID}'!")
except Exception as e:
    print(f"Note on weight loading: {e}. Initializing backbone structure for execution.")

# 3. Freeze All Base Backbone Parameters
for param in backbone.parameters():
    param.requires_grad = False

# 4. Wrap with Prefix Tuning Module
model = PrefixTunedHybridEmbedder(
    backbone=backbone, 
    num_prefix_tokens=CFG["num_prefix_tokens"], 
    prefix_mid_dim=CFG["prefix_mid_dim"]
)

# Ensure trainable parameters are in FP32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nPhase 7 Hybrid Encoder Prefix Tuning Summary:")
print(f"  - Total Parameters: {total_params:,}")
print(f"  - Trainable Prefix Parameters: {trainable_params:,} ({trainable_params/total_params*100:.3f}% of total)")
print(f"  - Frozen Base Parameters: {total_params - trainable_params:,}")

Fetching Phase 1 Checkpoint Weights from 'A-240242/legal-hybridencoder-checkpoints'...


model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded Phase 1 weights successfully from 'A-240242/legal-hybridencoder-checkpoints'!

Phase 7 Hybrid Encoder Prefix Tuning Summary:
  - Total Parameters: 72,233,984
  - Trainable Prefix Parameters: 5,133,824 (7.107% of total)
  - Frozen Base Parameters: 67,100,160


In [6]:
@torch.no_grad()
def evaluate_dev_pcr(model, dev_queries, dev_candidates, tokenizer, device, top_k=7):
    """
    Evaluates Prior Case Retrieval quality using Micro-F1@7 metric.
    """
    model.eval()
    
    cand_ids = list(dev_candidates.keys())
    cand_texts = [format_case_text(dev_candidates[cid]) for cid in cand_ids]
    
    cand_embeddings = []
    batch_size = 16
    for i in range(0, len(cand_texts), batch_size):
        batch_texts = cand_texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        emb = model(tok["input_ids"], attention_mask=tok["attention_mask"])
        cand_embeddings.append(emb.cpu())
        
    if len(cand_embeddings) == 0:
        return 0.0, 0.0, 0.0
        
    cand_embeddings = torch.cat(cand_embeddings, dim=0) # [N_cand, d]
    
    total_tp = 0
    total_pred = 0
    total_gold = 0
    
    for q in dev_queries:
        q_text = format_case_text(q.get("text", ""))
        gold_cands = set(q.get("relevant_candidates", []))
        
        tok = tokenizer(q_text, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        q_emb = model(tok["input_ids"], attention_mask=tok["attention_mask"]).cpu() # [1, d]
        
        sim_scores = torch.matmul(q_emb, cand_embeddings.T).squeeze(0) # [N_cand]
        top_indices = torch.topk(sim_scores, k=min(top_k, len(cand_ids))).indices.tolist()
        
        retrieved_ids = [cand_ids[idx] for idx in top_indices]
        
        tp = len(set(retrieved_ids).intersection(gold_cands))
        total_tp += tp
        total_pred += len(retrieved_ids)
        total_gold += len(gold_cands)
        
    precision = total_tp / max(1, total_pred)
    recall = total_tp / max(1, total_gold)
    micro_f1 = (2 * precision * recall) / max(1e-9, precision + recall)
    
    return micro_f1, precision, recall


CHECKPOINT_DIR = "./checkpoints_phase7_hybrid_prefix"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_hybrid_prefix_pcr.pt")

epochs = CFG["epochs"]
lr = CFG["lr"]

trainable_param_list = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_param_list, lr=lr, weight_decay=0.01)

best_micro_f1 = -1.0

print(f"\nCommencing Phase 7 Hybrid Encoder Prefix Tuning Contrastive Training ({epochs} Epochs)...")

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        q_ids = batch["query_input_ids"].to(device)
        q_mask = batch["query_attention_mask"].to(device)
        c_ids = batch["candidate_input_ids"].to(device)
        c_mask = batch["candidate_attention_mask"].to(device)
        
        # Forward pass for Queries & Candidates
        q_embeds = model(q_ids, attention_mask=q_mask) # [B, d]
        c_embeds = model(c_ids, attention_mask=c_mask) # [B, d]
        
        # Compute In-Batch Cosine Similarity Matrix
        logits = torch.matmul(q_embeds, c_embeds.T) / CFG["temperature"] # [B, B]
        labels = torch.arange(logits.size(0), device=device, dtype=torch.long)
        
        loss = F.cross_entropy(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Epoch {epoch+1}/{epochs} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / max(1, len(train_loader))
    
    # Dev Evaluation
    dev_f1, dev_p, dev_r = evaluate_dev_pcr(model, dev_queries, dev_candidates, tokenizer, device)
    print(f"\nEpoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Dev Micro-F1@7: {dev_f1:.4f} (P@7: {dev_p:.4f}, R@7: {dev_r:.4f})")

    if dev_f1 > best_micro_f1 or epoch == 0:
        best_micro_f1 = dev_f1
        trainable_state_dict = {
            k: v.cpu().clone() for k, v in model.state_dict().items() if v.requires_grad
        }
        
        checkpoint_data = {
            "epoch": epoch + 1,
            "best_micro_f1": best_micro_f1,
            "trainable_state_dict": trainable_state_dict
        }
        
        torch.save(checkpoint_data, BEST_CHECKPOINT_PATH)
        print(f"  [+] Saved New Best Checkpoint to '{BEST_CHECKPOINT_PATH}' (Dev Micro-F1@7: {best_micro_f1:.4f})")
        
    print("-" * 75)


Commencing Phase 7 Hybrid Encoder Prefix Tuning Contrastive Training (3 Epochs)...
Epoch 1/3 | Step 20/664 | Loss: 2.0634
Epoch 1/3 | Step 40/664 | Loss: 2.0771
Epoch 1/3 | Step 60/664 | Loss: 2.0327
Epoch 1/3 | Step 80/664 | Loss: 2.0891
Epoch 1/3 | Step 100/664 | Loss: 2.0812
Epoch 1/3 | Step 120/664 | Loss: 2.0756
Epoch 1/3 | Step 140/664 | Loss: 2.0712
Epoch 1/3 | Step 160/664 | Loss: 2.0119
Epoch 1/3 | Step 180/664 | Loss: 2.2584
Epoch 1/3 | Step 200/664 | Loss: 2.0695
Epoch 1/3 | Step 220/664 | Loss: 2.0784
Epoch 1/3 | Step 240/664 | Loss: 2.0617
Epoch 1/3 | Step 260/664 | Loss: 2.0315
Epoch 1/3 | Step 280/664 | Loss: 2.0682
Epoch 1/3 | Step 300/664 | Loss: 2.0526
Epoch 1/3 | Step 320/664 | Loss: 2.0712
Epoch 1/3 | Step 340/664 | Loss: 2.0292
Epoch 1/3 | Step 360/664 | Loss: 2.0689
Epoch 1/3 | Step 380/664 | Loss: 2.0598
Epoch 1/3 | Step 400/664 | Loss: 2.0606
Epoch 1/3 | Step 420/664 | Loss: 2.1080
Epoch 1/3 | Step 440/664 | Loss: 2.0211
Epoch 1/3 | Step 460/664 | Loss: 2.0619


In [7]:
print("Reloading Best Hybrid Encoder Prefix Tuning Checkpoint for Test Set Retrieval...")

if os.path.exists(BEST_CHECKPOINT_PATH):
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Successfully reloaded checkpoint from Epoch {checkpoint['epoch']} (Best Dev Micro-F1@7: {checkpoint['best_micro_f1']:.4f})\n")

model.eval()

# 1. Encode all Candidate Cases in the Test Candidate Pool
test_cand_ids = list(test_candidates.keys())
test_cand_texts = [format_case_text(test_candidates[cid]) for cid in test_cand_ids]

print(f"Encoding {len(test_cand_ids)} Test Candidate Case Documents...")

cand_embeddings_list = []
batch_size = 16

with torch.no_grad():
    for i in range(0, len(test_cand_texts), batch_size):
        batch_texts = test_cand_texts[i:i+batch_size]
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        emb = model(tok["input_ids"], attention_mask=tok["attention_mask"])
        cand_embeddings_list.append(emb.cpu())

test_cand_matrix = torch.cat(cand_embeddings_list, dim=0) # [N_test_cand, d]

# 2. Retrieve Top-7 Candidates for Each Test Query
print(f"Ranking Candidate Pool for {len(test_queries)} Test Queries...")

predictions = {}

for q in test_queries:
    q_id = str(q.get("id"))
    q_text = format_case_text(q.get("text", ""))
    
    with torch.no_grad():
        tok = tokenizer(q_text, truncation=True, max_length=CFG["max_seq_len"], return_tensors="pt").to(device)
        q_emb = model(tok["input_ids"], attention_mask=tok["attention_mask"]).cpu() # [1, d]
        
    scores = torch.matmul(q_emb, test_cand_matrix.T).squeeze(0) # [N_test_cand]
    
    # Retrieve top 7 candidate IDs
    top_k = min(7, len(test_cand_ids))
    top_indices = torch.topk(scores, k=top_k).indices.tolist()
    
    retrieved_cands = [str(test_cand_ids[idx]) for idx in top_indices]
    
    # Fallback to pad to exactly 7 candidate IDs if candidate pool is small
    while len(retrieved_cands) < 7:
        for cid in test_cand_ids:
            cid_str = str(cid)
            if cid_str not in retrieved_cands:
                retrieved_cands.append(cid_str)
            if len(retrieved_cands) == 7:
                break
                
    predictions[q_id] = retrieved_cands[:7]

print(f"\nGenerated predictions for {len(predictions)} test queries.")

Reloading Best Hybrid Encoder Prefix Tuning Checkpoint for Test Set Retrieval...
Successfully reloaded checkpoint from Epoch 3 (Best Dev Micro-F1@7: 0.0200)

Encoding 1727 Test Candidate Case Documents...
Ranking Candidate Pool for 237 Test Queries...

Generated predictions for 237 test queries.


In [8]:
# Output serialization and zip packaging according to Phase 7 requirements
output_dir = "./phase7_hybrid_prefix_pcr"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 7 Hybrid Encoder Prefix Tuning Prior Case Retrieval Complete! Generated rankings for {len(predictions)} queries.")
print(f"Submission file saved to: {sub_zip_path}")

if DEBUG:
    print("\n[DEBUG] Sample Prediction Output (First 2 Queries):")
    sample_keys = list(predictions.keys())[:2]
    sample_dict = {k: predictions[k] for k in sample_keys}
    print(json.dumps(sample_dict, indent=2))


Phase 7 Hybrid Encoder Prefix Tuning Prior Case Retrieval Complete! Generated rankings for 237 queries.
Submission file saved to: ./phase7_hybrid_prefix_pcr/submission.zip
